# Exploring the test dataset

In [2]:
"""
Video metadata inspector for cricket ball tracking dataset.
Run this on Kaggle: python inspect_videos.py
"""

import subprocess
import json
import os
import sys
from pathlib import Path

# ── Configure paths ────────────────────────────────────────────────────────────
DATASET_DIR = "/kaggle/input/cricketballtracking"  # adjust if needed
FALLBACK_DIRS = [
    "/kaggle/input/datasets/vaibhavsonkar/cricketballtracking",
    "/kaggle/input/cricketballtracking",
    ".",
]

def find_dataset_dir():
    for d in FALLBACK_DIRS:
        p = Path(d)
        if p.exists():
            videos = list(p.glob("**/*.mp4")) + list(p.glob("**/*.mov"))
            if videos:
                return p
    return None

def probe_video(path: Path) -> dict:
    """Use ffprobe to extract key video metadata."""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_streams",
        "-show_format",
        str(path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return {"error": result.stderr.strip()}

    data = json.loads(result.stdout)
    video_stream = next(
        (s for s in data.get("streams", []) if s.get("codec_type") == "video"),
        None
    )
    if not video_stream:
        return {"error": "No video stream found"}

    # Parse FPS (can be a fraction string like "30000/1001")
    def parse_fps(fps_str):
        if not fps_str or fps_str == "0/0":
            return None
        parts = fps_str.split("/")
        if len(parts) == 2:
            num, den = int(parts[0]), int(parts[1])
            return round(num / den, 3) if den != 0 else None
        return float(fps_str)

    fps_raw = video_stream.get("r_frame_rate", "")
    avg_fps_raw = video_stream.get("avg_frame_rate", "")
    fps = parse_fps(fps_raw)
    avg_fps = parse_fps(avg_fps_raw)

    width  = video_stream.get("width", "?")
    height = video_stream.get("height", "?")
    codec  = video_stream.get("codec_name", "?")
    pix_fmt = video_stream.get("pix_fmt", "?")

    duration = float(video_stream.get("duration")
                     or data.get("format", {}).get("duration", 0) or 0)

    nb_frames = video_stream.get("nb_frames")
    if nb_frames and nb_frames != "N/A":
        total_frames = int(nb_frames)
    elif fps and duration:
        total_frames = int(fps * duration)
    else:
        total_frames = "?"

    size_bytes = int(data.get("format", {}).get("size", 0) or 0)
    size_mb = round(size_bytes / 1e6, 1)

    return {
        "resolution": f"{width}×{height}",
        "width": width,
        "height": height,
        "fps": fps,
        "avg_fps": avg_fps,
        "codec": codec,
        "pix_fmt": pix_fmt,
        "duration_s": round(duration, 2),
        "total_frames": total_frames,
        "size_mb": size_mb,
    }

def classify_resolution(w, h):
    mp = w * h
    if   mp >= 3840 * 2160: return "4K"
    elif mp >= 2560 * 1440: return "2K/QHD"
    elif mp >= 1920 * 1080: return "FHD (1080p)"
    elif mp >= 1280 *  720: return "HD (720p)"
    elif mp >=  854 *  480: return "SD (480p)"
    else:                   return "Low-res"

def strategy_hint(fps, w, h):
    hints = []
    if fps and fps >= 120:
        hints.append("Super slow-mo — TrackNet input @ native FPS; excellent for motion study")
    elif fps and fps >= 60:
        hints.append("High FPS — ideal for TrackNet (3-frame stacking keeps blur low)")
    elif fps and fps >= 24:
        hints.append("Standard broadcast — expect motion blur; SAHI + TrackNetV3 recommended")
    else:
        hints.append("Very low FPS — ball may skip multiple positions per frame")

    if w and h:
        mp = w * h
        if mp >= 1920 * 1080:
            hints.append("High-res — SAHI tiling highly beneficial (ball ~5-15px)")
        else:
            hints.append("Lower-res — ball even smaller; increase imgsz, use TrackNet heatmaps")

    return " | ".join(hints)

def main():
    dataset_dir = find_dataset_dir()
    if not dataset_dir:
        print("Could not find dataset directory. Update FALLBACK_DIRS in the script.")
        sys.exit(1)

    print(f"\n Scanning: {dataset_dir}\n")

    extensions = ["*.mp4", "*.mov", "*.avi", "*.mkv", "*.MP4", "*.MOV"]
    video_files = []
    for ext in extensions:
        video_files.extend(dataset_dir.glob(ext))
        video_files.extend(dataset_dir.glob(f"**/{ext}"))

    video_files = sorted(set(video_files), key=lambda p: p.name)

    if not video_files:
        print("No video files found.")
        sys.exit(1)

    print(f"Found {len(video_files)} video files\n")
    print("=" * 100)

    results = []
    for vf in video_files:
        info = probe_video(vf)
        info["filename"] = vf.name
        results.append(info)

    # ── Print table ────────────────────────────────────────────────────────────
    header = f"{'File':<15} {'Resolution':<14} {'Class':<12} {'FPS':>8} {'AvgFPS':>8} {'Duration':>9} {'Frames':>8} {'Size':>8} {'Codec':<8}"
    print(header)
    print("-" * len(header))

    for r in results:
        if "error" in r:
            print(f"{r['filename']:<15}  ERROR: {r['error']}")
            continue
        cls = classify_resolution(r.get("width", 0), r.get("height", 0))
        print(
            f"{r['filename']:<15} "
            f"{r['resolution']:<14} "
            f"{cls:<12} "
            f"{str(r.get('fps', '?')):>8} "
            f"{str(r.get('avg_fps', '?')):>8} "
            f"{str(r.get('duration_s', '?')) + 's':>9} "
            f"{str(r.get('total_frames', '?')):>8} "
            f"{str(r.get('size_mb', '?')) + 'MB':>8} "
            f"{r.get('codec', '?'):<8}"
        )

    # ── Per-file strategy hints ────────────────────────────────────────────────
    print("\n" + "=" * 100)
    print("STRATEGY HINTS PER FILE")
    print("=" * 100)
    for r in results:
        if "error" in r:
            continue
        hint = strategy_hint(r.get("fps"), r.get("width"), r.get("height"))
        print(f"\n{r['filename']}")
        print(f"   {hint}")

    # ── Summary stats ──────────────────────────────────────────────────────────
    valid = [r for r in results if "error" not in r]
    if valid:
        all_fps  = [r["fps"] for r in valid if r.get("fps")]
        all_res  = [(r["width"], r["height"]) for r in valid if r.get("width")]
        total_dur = sum(r.get("duration_s", 0) for r in valid)
        total_frames = sum(r.get("total_frames", 0) for r in valid if isinstance(r.get("total_frames"), int))

        print("\n" + "=" * 100)
        print("SUMMARY")
        print("=" * 100)
        print(f"  Total videos       : {len(valid)}")
        print(f"  Total duration     : {round(total_dur, 1)}s  ({round(total_dur/60, 1)} min)")
        print(f"  Total frames (est) : {total_frames:,}")
        print(f"  FPS range          : {min(all_fps)} – {max(all_fps)}")
        if all_res:
            max_res = max(all_res, key=lambda x: x[0]*x[1])
            min_res = min(all_res, key=lambda x: x[0]*x[1])
            print(f"  Resolution range   : {min_res[0]}×{min_res[1]}  →  {max_res[0]}×{max_res[1]}")
        print()

    # ── Save JSON ──────────────────────────────────────────────────────────────
    out_path = Path("/kaggle/working/video_metadata.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Metadata saved to: {out_path}\n")

if __name__ == "__main__":
    main()


 Scanning: /kaggle/input/datasets/vaibhavsonkar/cricketballtracking

Found 15 video files

File            Resolution     Class             FPS   AvgFPS  Duration   Frames     Size Codec   
--------------------------------------------------------------------------------------------------
1.mp4           1920×1080      FHD (1080p)      25.0     25.0     1.24s       31    1.8MB mpeg4   
10.mov          2560×1440      2K/QHD           60.0   42.526     4.71s      202   14.2MB h264    
11.mov          2558×1444      2K/QHD           60.0    44.81     3.95s      177   15.1MB h264    
12.mov          2560×1440      2K/QHD           60.0   47.119     2.93s      139    8.9MB h264    
13.mov          2560×1440      2K/QHD           60.0   44.315     4.01s      178   12.8MB h264    
14.mov          2558×1442      2K/QHD           60.0     49.0      4.0s      196   13.4MB h264    
15.mov          2560×1440      2K/QHD           60.0   44.016     4.03s      179   14.6MB h264    
2.mov           2

# Dataset pull (Roboflow → Kaggle)

In [29]:
!pip install roboflow -q
import shutil
from roboflow import Roboflow

API_KEY = "Sf0csHVN6JzV4pukGaCs"
rf = Roboflow(api_key=API_KEY)

DATASETS = [
    ("originals-workspace-nz2s0", "ball-tracking-axppe-lhlw3",              1, "ds1"),
    ("originals-workspace-nz2s0", "ball-tracking-ncwes-21qy2",              1, "ds2"),
    ("originals-workspace-nz2s0", "ball-tracking-ub0zo-sep7w",              1, "ds3"),
    ("originals-workspace-nz2s0", "cricket-spectating-ball-tracking-piwcs", 1, "ds4"),
    ("originals-workspace-nz2s0", "cricket-ball-detection-zplao-i2p6r",     1, "ds5"),
    ("originals-workspace-nz2s0", "cricket-ball-nffxs-72nao",               1, "ds6"),
    ("originals-workspace-nz2s0", "final-edgefleet-project-u7zlz-wuewk",   1, "ds7"),
    ("originals-workspace-nz2s0", "my-first-project-08fti-2uojr",           1, "ds8"),
    ("originals-workspace-nz2s0", "pg1-pi8fe-1kg13",                        1, "ds9"),
]

for workspace, project_slug, version, alias in DATASETS:
    print(f"Downloading {alias}: {project_slug}...")
    shutil.rmtree(f"/kaggle/working/raw/{alias}", ignore_errors=True)
    proj = rf.workspace(workspace).project(project_slug)
    proj.version(version).download("yolov8", location=f"/kaggle/working/raw/{alias}")
    print(f"  ✓ {alias} done")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds1 in yolov8:: 100%|██████████| 1021/1021 [00:00<00:00, 11442.61it/s]

  ✓ ds1 done
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds2 in yolov8:: 100%|██████████| 593/593 [00:00<00:00, 7616.60it/s]


...

loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds8 in yolov8:: 100%|██████████| 1729/1729 [00:00<00:00, 2647.51it/s]


  ✓ ds8 done
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds9 in yolov8:: 100%|██████████| 1507/1507 [00:00<00:00, 9631.47it/s]

  ✓ ds9 done


In [16]:
# # Cell: Inspect your workspace to get exact project slugs
# from roboflow import Roboflow
# rf = Roboflow(api_key="Sf0csHVN6JzV4pukGaCs")
# workspace = rf.workspace("originals-workspace-nz2s0")

# # projects() returns a list of strings (slugs directly)
# projects = workspace.projects()
# print("All project slugs in workspace:")
# for p in projects:
#     print(f"  {p}")

In [17]:
# # For ds4, ds8
# import shutil, os

# for alias, slug in [
#     ("ds4", "cricket-spectating-ball-tracking-piwcs"),
#     ("ds8", "my-first-project-08fti-2uojr"),
# ]:
#     print(f"\nRe-downloading {alias}: {slug}...")
#     shutil.rmtree(f"/kaggle/working/raw/{alias}", ignore_errors=True)
#     proj = rf.workspace("originals-workspace-nz2s0").project(slug)
#     proj.version(1).download("yolov8", location=f"/kaggle/working/raw/{alias}")
    
#     print(f"Files in {alias}:")
#     for root, dirs, files in os.walk(f"/kaggle/working/raw/{alias}"):
#         if files:
#             print(f"  {root}: {len(files)} files")


Re-downloading ds4: cricket-spectating-ball-tracking-piwcs...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds4 in yolov8:: 100%|██████████| 2495/2495 [00:00<00:00, 3226.45it/s]


Files in ds4:
  /kaggle/working/raw/ds4: 3 files
  /kaggle/working/raw/ds4/test/images: 125 files
  /kaggle/working/raw/ds4/test/labels: 125 files
  /kaggle/working/raw/ds4/valid/images: 248 files
  /kaggle/working/raw/ds4/valid/labels: 248 files
  /kaggle/working/raw/ds4/train/images: 872 files
  /kaggle/working/raw/ds4/train/labels: 872 files

Re-downloading ds8: my-first-project-08fti-2uojr...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds8 in yolov8:: 100%|██████████| 1729/1729 [00:00<00:00, 2693.30it/s]

Files in ds8:
  /kaggle/working/raw/ds8: 3 files
  /kaggle/working/raw/ds8/train/images: 863 files
  /kaggle/working/raw/ds8/train/labels: 863 files


In [19]:
# import shutil, os
# from pathlib import Path
# from roboflow import Roboflow

# rf = Roboflow(api_key="Sf0csHVN6JzV4pukGaCs")

# # Re-download ds3 and ds8
# for alias, slug in [
#     ("ds3", "ball-tracking-ub0zo-sep7w"),
#     ("ds8", "my-first-project-08fti-2uojr"),
# ]:
#     print(f"Re-downloading {alias}...")
#     shutil.rmtree(f"/kaggle/working/raw/{alias}", ignore_errors=True)
#     rf.workspace("originals-workspace-nz2s0").project(slug).version(1).download(
#         "yolov8", location=f"/kaggle/working/raw/{alias}"
#     )
#     print(f"  ✓ {alias} done")

# # Inspect raw class distribution BEFORE any remap
# print("\n--- Raw class distribution after fresh download ---")
# for ds_name in ["ds3", "ds8"]:
#     ds_root = Path(f"/kaggle/working/raw/{ds_name}")
#     class_counts = {}
#     for label_file in ds_root.rglob("labels/*.txt"):
#         for line in open(label_file).read().strip().splitlines():
#             if not line.strip():
#                 continue
#             cls = int(line.split()[0])
#             class_counts[cls] = class_counts.get(cls, 0) + 1
#     print(f"\n{ds_name} class distribution:")
#     for cls_id, count in sorted(class_counts.items()):
#         print(f"  class {cls_id}: {count} annotations")

Re-downloading ds3...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds3 in yolov8:: 100%|██████████| 6009/6009 [00:00<00:00, 11121.75it/s]


  ✓ ds3 done
Re-downloading ds8...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw/ds8 in yolov8:: 100%|██████████| 1729/1729 [00:00<00:00, 2679.28it/s]


  ✓ ds8 done

--- Raw class distribution after fresh download ---

ds3 class distribution:
  class 0: 560 annotations
  class 1: 277 annotations
  class 2: 806 annotations
  class 3: 700 annotations
  class 4: 501 annotations
  class 5: 627 annotations
  class 6: 875 annotations

ds8 class distribution:
  class 0: 882 annotations
  class 1: 18 annotations


In [30]:
from pathlib import Path

BALL_CLASSES = {
    "ds1": {0},
    "ds2": {0},
    "ds3": {2, 4},  # ball + cricket ball
    "ds4": {0},
    "ds5": {0},
    "ds6": {0},
    "ds7": {0},
    "ds8": {1},     # class 0 is junk
    "ds9": {0},
}

for ds_name, ball_ids in BALL_CLASSES.items():
    ds_root = Path(f"/kaggle/working/raw/{ds_name}")
    fixed = dropped = 0
    for label_file in ds_root.rglob("labels/*.txt"):
        lines = open(label_file).read().strip().splitlines()
        new_lines = []
        for line in lines:
            if not line.strip():
                continue
            parts = line.split()
            cls = int(parts[0])
            if cls in ball_ids:
                parts[0] = "0"
                new_lines.append(" ".join(parts))
                fixed += 1
            else:
                dropped += 1
        open(label_file, "w").write("\n".join(new_lines))
    print(f"{ds_name}: remapped {fixed} ball annotations, dropped {dropped} non-ball")

ds1: remapped 492 ball annotations, dropped 0 non-ball
ds2: remapped 282 ball annotations, dropped 0 non-ball
ds3: remapped 1307 ball annotations, dropped 3039 non-ball
ds4: remapped 1251 ball annotations, dropped 0 non-ball
ds5: remapped 638 ball annotations, dropped 330 non-ball
ds6: remapped 79 ball annotations, dropped 0 non-ball
ds7: remapped 373 ball annotations, dropped 0 non-ball
ds8: remapped 18 ball annotations, dropped 882 non-ball
ds9: remapped 751 ball annotations, dropped 0 non-ball


In [31]:
# Cell 3: Merge into unified dataset
DEST = Path("/kaggle/working/dataset")

for split in ["train", "valid", "test"]:
    (DEST / split / "images").mkdir(parents=True, exist_ok=True)
    (DEST / split / "labels").mkdir(parents=True, exist_ok=True)

for _, _, _, alias in DATASETS:
    src = Path(f"/kaggle/working/raw/{alias}")
    for split in ["train", "valid", "test"]:
        for ext in ["images", "labels"]:
            src_dir = src / split / ext
            dst_dir = DEST / split / ext
            if not src_dir.exists():
                continue
            for f in src_dir.iterdir():
                dst_file = dst_dir / f"{alias}_{f.name}"
                shutil.copy2(f, dst_file)

print("Merge complete:")
for split in ["train", "valid", "test"]:
    n = len(list((DEST / split / "images").iterdir()))
    print(f"  {split}: {n} images")

Merge complete:
  train: 6303 images
  valid: 1218 images
  test: 562 images


In [23]:
# Cell 4: Inspect class names across all datasets before writing yaml
import os

for _, _, _, alias in DATASETS:
    yaml_candidates = list(Path(f"/kaggle/working/raw/{alias}").glob("*.yaml"))
    if yaml_candidates:
        print(f"\n--- {alias} ---")
        print(open(yaml_candidates[0]).read())


--- ds1 ---
names:
- ball
nc: 1
roboflow:
  license: CC BY 4.0
  project: ball-tracking-axppe-lhlw3
  url: https://universe.roboflow.com/originals-workspace-nz2s0/ball-tracking-axppe-lhlw3/dataset/1
  version: 1
  workspace: originals-workspace-nz2s0
test: ../test/images
train: ../train/images
val: ../valid/images


--- ds2 ---
names:
- ball
nc: 1
roboflow:
  license: Private
  project: ball-tracking-ncwes-21qy2
  url: https://universe.roboflow.com/originals-workspace-nz2s0/ball-tracking-ncwes-21qy2/dataset/1
  version: 1
  workspace: originals-workspace-nz2s0
test: ../test/images
train: ../train/images
val: ../valid/images


--- ds3 ---
names:
- Batsman
- No ball
- ball
- bat
- cricket ball
- pitch
- umpire_no_ball
nc: 7
roboflow:
  license: CC BY 4.0
  project: ball-tracking-ub0zo-sep7w
  url: https://universe.roboflow.com/originals-workspace-nz2s0/ball-tracking-ub0zo-sep7w/dataset/1
  version: 1
  workspace: originals-workspace-nz2s0
test: ../test/images
train: ../train/images
val:

In [8]:
# import shutil
# shutil.rmtree("/kaggle/working/dataset", ignore_errors=True)
# print("Cleared.")

In [24]:
# Remove image+label pairs where label file is empty or missing
from pathlib import Path

DEST = Path("/kaggle/working/dataset")
removed = 0

for split in ["train", "valid", "test"]:
    img_dir = DEST / split / "images"
    lbl_dir = DEST / split / "labels"
    
    for img_file in list(img_dir.iterdir()):
        stem = img_file.stem
        lbl_file = lbl_dir / f"{stem}.txt"
        
        # Remove if label missing or empty
        if not lbl_file.exists() or lbl_file.stat().st_size == 0:
            img_file.unlink()
            lbl_file.unlink(missing_ok=True)
            removed += 1

print(f"Removed {removed} image/label pairs with no ball annotations")

print("\nFinal counts:")
for split in ["train", "valid", "test"]:
    n = len(list((DEST / split / "images").iterdir()))
    print(f"  {split}: {n} images")

Removed 2220 image/label pairs with no ball annotations

Final counts:
  train: 3814 images
  valid: 815 images
  test: 371 images


In [25]:
import yaml
from pathlib import Path

DEST = Path("/kaggle/working/dataset")

yaml_content = {
    "path"  : str(DEST),
    "train" : "train/images",
    "val"   : "valid/images",
    "test"  : "test/images",
    "nc"    : 1,
    "names" : ["ball"]
}

with open(DEST / "data.yaml", "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("data.yaml written:")
print(open(DEST / "data.yaml").read())

data.yaml written:
names:
- ball
nc: 1
path: /kaggle/working/dataset
test: test/images
train: train/images
val: valid/images



# Training Yolov8

In [28]:
# Step 2: Fine-tune YOLOv8x
!pip install ultralytics -q

from ultralytics import YOLO
from pathlib import Path

MODEL    = "yolov8x.pt"       # pretrained COCO weights
DATA     = "/kaggle/working/dataset/data.yaml"
EPOCHS   = 45
IMGSZ    = 1280
BATCH    = 8                  # safe for T4/P100 at imgsz=1280
PROJECT  = "/kaggle/working/runs"
NAME     = "cricket_ball_v1"

model = YOLO(MODEL)

results = model.train(
    data       = DATA,
    epochs     = EPOCHS,
    imgsz      = IMGSZ,
    batch      = BATCH,
    project    = PROJECT,
    name       = NAME,
    pretrained = True,
    optimizer  = "AdamW",
    lr0        = 0.001,
    lrf        = 0.01,
    momentum   = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    cos_lr     = True,
    mosaic     = 1.0,
    mixup      = 0.1,
    fliplr     = 0.5,
    flipud     = 0.1,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 5.0,         # slight rotation — ball is round so safe
    translate  = 0.1,
    scale      = 0.5,
    patience   = 15,          # early stop if no improvement for 15 epochs
    save       = True,
    save_period = 10,
    val        = True,
    device     = [0,1],
    workers    = 8,
    verbose    = True,
)

print(f"\nBest weights: {PROJECT}/{NAME}/weights/best.pt")

Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=5.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cricket_ball_v110, nbs=64, n

W0402 20:16:24.340000 498 torch/distributed/elastic/agent/server/api.py:739] Received 2 death signal, shutting down workers
W0402 20:16:24.342000 498 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 501 closing signal SIGINT
W0402 20:16:24.343000 498 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 502 closing signal SIGINT
[rank1]: Traceback (most recent call last):
[rank1]:   File "/root/.config/Ultralytics/DDP/_temp_e8lllyn6134867038188608.py", line 19, in <module>
[rank1]:     results = trainer.train()
[rank1]:               ^^^^^^^^^^^^^^^
[rank1]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 244, in train
[rank1]:     self._do_train()
[rank1]:   File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py", line 445, in _do_train
[rank1]:     self.scaler.scale(self.loss).backward()
[rank1]:   File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 630, in backward
[ran

       1/50      11.8G      2.302      8.006      1.828          4       1280: 21% ━━╸───────── 102/477 1.4s/it 2:25<8:39


KeyboardInterrupt: 

In [ ]:
# After training — check results   
import pandas as pd 

results_csv = f"{PROJECT}/{NAME}/results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print(df[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(10).to_string())

# CFR (Constant Frame Rate) conversion